In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import base64

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
train_clean_dir     = '/kaggle/input/competitions/watermark-removal-aicc-round-5/dataset/train/clean'
train_watermarked_dir = '/kaggle/input/competitions/watermark-removal-aicc-round-5/dataset/train/watermarked'
test_dir            = '/kaggle/input/competitions/watermark-removal-aicc-round-5/dataset/test'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

In [ ]:
# ── ТРАНСФОРМАЦИИ ──────────────────────────────────────────────────────────────
# ВАЖНО: НЕТ двойной нормализации — только A.Normalize, делим на 255 один раз
# A.Normalize уже переводит в float и нормирует → делить на 255 БОЛЬШЕ НЕ НУЖНО

IMG_SIZE = 224

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=20, p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.4),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], additional_targets={'image2': 'image'})

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], additional_targets={'image2': 'image'})

test_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [ ]:
# ── ДАТАСЕТЫ ───────────────────────────────────────────────────────────────────

class TrainDataset(Dataset):
    def __init__(self, clean_dir, water_dir, transform):
        self.clean_dir = clean_dir
        self.water_dir = water_dir
        self.transform = transform
        self.names = sorted(os.listdir(clean_dir))

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        imgw = np.array(Image.open(os.path.join(self.water_dir, name)).convert('RGB'))
        imgc = np.array(Image.open(os.path.join(self.clean_dir, name)).convert('RGB'))
        aug = self.transform(image=imgw, image2=imgc)
        # НЕТ деления на 255 — A.Normalize уже всё сделала
        return aug['image'].float(), aug['image2'].float()


class TestDataset(Dataset):
    def __init__(self, test_dir, transform):
        self.test_dir = test_dir
        self.transform = transform
        self.names = sorted(os.listdir(test_dir))

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        img = np.array(Image.open(os.path.join(self.test_dir, name)).convert('RGB'))
        aug = self.transform(image=img)
        return aug['image'].float(), name

In [ ]:
# ── SPLIT TRAIN/VAL ────────────────────────────────────────────────────────────
full_dataset = TrainDataset(train_clean_dir, train_watermarked_dir, train_transform)

val_size  = int(0.1 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_ds, val_ds = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# У val_ds трансформ должен быть val_transform — пересоздаём датасет для val
val_ds_clean = TrainDataset(train_clean_dir, train_watermarked_dir, val_transform)
val_indices  = val_ds.indices
val_ds       = torch.utils.data.Subset(val_ds_clean, val_indices)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

In [ ]:
# ── МОДЕЛЬ ─────────────────────────────────────────────────────────────────────
!pip install -q segmentation-models-pytorch
import segmentation_models_pytorch as smp

# Используем EfficientNet-B4 как энкодер — хороший баланс качества и скорости
model = smp.Unet(
    encoder_name='efficientnet-b4',
    encoder_weights='imagenet',
    in_channels=3,
    classes=3,
    activation=None,   # без sigmoid/softmax — сами контролируем выход
).to(device)

# ── Размораживаем ВСЕ параметры (или хотя бы decoder + head) ──────────────────
# Вариант А: полностью trainable
for param in model.parameters():
    param.requires_grad = True

# Вариант Б (если мало VRAM): заморозить первые слои encoder
# for name, param in model.encoder.named_parameters():
#     if 'blocks.0' in name or 'blocks.1' in name:
#         param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {trainable:,}')

In [ ]:
# ── LOSS + OPTIMIZER + SCHEDULER ──────────────────────────────────────────────
# Комбинированный loss: MSE (для метрики) + L1 (устойчив к выбросам)
# Perceptual loss можно добавить позже как трюк

mse_loss = nn.MSELoss()
l1_loss  = nn.L1Loss()

def criterion(pred, target):
    return 0.5 * mse_loss(pred, target) + 0.5 * l1_loss(pred, target)

epochs    = 30
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

In [ ]:
# ── ТРЕНИРОВОЧНЫЙ ЦИКЛ ─────────────────────────────────────────────────────────
# denormalize нужен чтобы посчитать честный MSE в [0,1]
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(device)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(device)

def denorm(x):
    return (x * STD + MEAN).clamp(0, 1)

best_val_mse = float('inf')

for epoch in range(epochs):
    # ── Train ──
    model.train()
    train_loss = 0.0
    for imgw, imgc in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [train]'):
        imgw, imgc = imgw.to(device), imgc.to(device)
        optimizer.zero_grad()
        out  = model(imgw)
        loss = criterion(out, imgc)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()

    # ── Val MSE в [0,1] ──
    model.eval()
    val_mse = 0.0
    with torch.no_grad():
        for imgw, imgc in val_loader:
            imgw, imgc = imgw.to(device), imgc.to(device)
            out = model(imgw)
            # считаем MSE в оригинальном [0,1] пространстве
            val_mse += mse_loss(denorm(out), denorm(imgc)).item()

    train_loss /= len(train_loader)
    val_mse    /= len(val_loader)
    # перевод в масштаб соревнования: пиксели в [0,255] → MSE * 255^2 / (255^2) = val_mse * 255^2
    # но оценка [0,1] → просто val_mse
    print(f'Epoch {epoch+1:02d} | train_loss={train_loss:.5f} | val_MSE={val_mse:.6f} (×255²={val_mse*255**2:.1f})')

    if val_mse < best_val_mse:
        best_val_mse = val_mse
        torch.save(model.state_dict(), '/kaggle/working/best_model.pth')
        print('  ✓ saved best model')

In [ ]:
# ── ИНФЕРЕНС ───────────────────────────────────────────────────────────────────
model.load_state_dict(torch.load('/kaggle/working/best_model.pth'))
model.eval()

test_dataset = TestDataset(test_dir, test_transform)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

output_dir = '/kaggle/working/predictions'
os.makedirs(output_dir, exist_ok=True)

with torch.no_grad():
    for imgs, names in tqdm(test_loader, desc='Inference'):
        imgs = imgs.to(device)
        out  = model(imgs)              # нормированный выход
        out  = denorm(out)              # → [0, 1]
        out  = (out * 255).clamp(0, 255).byte().cpu()

        for i, name in enumerate(names):
            arr = out[i].permute(1, 2, 0).numpy()
            # ресайзим обратно до 64×64 — именно такой ожидает оценка
            img_pil = Image.fromarray(arr).resize((64, 64), Image.LANCZOS)
            img_pil.save(os.path.join(output_dir, name))

print('Done! Saved', len(os.listdir(output_dir)), 'images')

In [ ]:
# ── SUBMISSION CSV ─────────────────────────────────────────────────────────────
# subtaskID=1 обязателен согласно формату задачи!

def generate_csv(image_folder, output_csv):
    files = sorted([f for f in os.listdir(image_folder) if f.endswith('.png')])
    rows = []
    for filename in files:
        filepath = os.path.join(image_folder, filename)
        with open(filepath, 'rb') as f:
            encoded = base64.b85encode(f.read()).decode('utf-8')
        rows.append({'datapointID': filename, 'subtaskID': 1, 'answer': encoded})

    df = pd.DataFrame(rows, columns=['datapointID', 'subtaskID', 'answer'])
    df.to_csv(output_csv, index=False)
    print(f'Submission saved: {output_csv} ({len(df)} rows)')
    return df

df = generate_csv(output_dir, '/kaggle/working/submission.csv')
df.head()

In [ ]:
# ── ВИЗУАЛИЗАЦИЯ результата ────────────────────────────────────────────────────
sample_name = sorted(os.listdir(output_dir))[0]
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(Image.open(os.path.join(test_dir, sample_name)))
axes[0].set_title('Watermarked (input)')
axes[1].imshow(Image.open(os.path.join(output_dir, sample_name)))
axes[1].set_title('Restored (output)')
plt.tight_layout()
plt.show()